<a href="https://colab.research.google.com/github/Ashi743/PYTORCH-DEEP-LEARNING/blob/main/CNN_VGG16_CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("playatanu/cifar10")

print("Path to dataset files:", path)

100%|██████████| 52.8M/52.8M [00:00<00:00, 184MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/playatanu/cifar10/versions/1


In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import os

image_base_path = os.path.join(path, "CIFAR10")

train_dir_pt = os.path.join(image_base_path, 'train')
test_dir_pt = os.path.join(image_base_path, 'test')

print(f"PyTorch Training data directory: {train_dir_pt}")
print(f"PyTorch Testing data directory: {test_dir_pt}")


PyTorch Training data directory: /root/.cache/kagglehub/datasets/playatanu/cifar10/versions/1/CIFAR10/train
PyTorch Testing data directory: /root/.cache/kagglehub/datasets/playatanu/cifar10/versions/1/CIFAR10/test


In [3]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


**ll pre-trained models expect input images normalized in the same way, i.e. mini-batches of 3-channel RGB images of shape (3 x H x W), where H and W are expected to be at least 224. The images have to be loaded in to a range of [0, 1] and then normalized using mean = [0.485, 0.456, 0.406] and std = [0.229, 0.224, 0.225].**

In [4]:
from torchvision.transforms import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [5]:
IMG_HEIGHT_PT = 32
IMG_WIDTH_PT = 32
BATCH_SIZE_PT = 64 #32 takes time


###ImageFolder is a PyTorch dataset class designed to load images from a directory structure where each subfolder represents a class label.

from torch.utils.data import DataLoader, Dataset
### CUSTOM DATASET AND DATALOADER
# Load the training dataset using ImageFolder
train_dataset = torchvision.datasets.ImageFolder(
    root=train_dir_pt,
    transform= transform_train
)

# Load the testing dataset using ImageFolder
test_dataset = torchvision.datasets.ImageFolder(
    root=test_dir_pt,
    transform=transform_train
)

### DataLoaders
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE_PT,
    shuffle=True,
    num_workers=2,
    pin_memory=True # to copy Tensors into CUDA pinned memory before returning them
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE_PT,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

class_names_pt = train_dataset.classes
print(f"Class names: {class_names_pt}")

Class names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [6]:
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of testing samples: {len(test_dataset)}")
print(f"Number of classes: {len(class_names_pt)}")
print(f"Class names: {class_names_pt}")

image, label = train_dataset[0]
print(f"Shape of first training image: {image.shape}")
print(f"Label of first training image: {class_names_pt[label]}")

Number of training samples: 50000
Number of testing samples: 10000
Number of classes: 10
Class names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Shape of first training image: torch.Size([3, 224, 224])
Label of first training image: airplane


### Defining a Convolutional Neural Network (CNN)  with Transfer Learing in PyTorch

In [7]:
# fetch pretrained models
import torchvision.models as models

In [8]:
vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 168MB/s]


In [9]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [10]:
#to freeze pretrained feature parameters
for params in vgg16.parameters():
  params.requires_grad = False

In [11]:
import torch.nn as nn
vgg16.classifier = nn.Sequential(
    nn.Linear(25088, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 64),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(64, 10)
)

In [12]:
vgg16= vgg16.to(device)

In [13]:
learning_rate = 0.001
epochs= 10

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(vgg16.classifier.parameters(), lr= learning_rate)

In [15]:
import time

# for training loop
for epoch in range(epochs):
  start_time = time.time() # Start timing for the epoch
  total_epoch_loss = 0
  vgg16.train()
  for images, labels in train_dataloader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    outputs = vgg16(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    total_epoch_loss += loss.item()
  avg_loss= total_epoch_loss / len(train_dataloader)
  end_time = time.time() # End timing for the epoch
  epoch_duration = end_time - start_time # Calculate duration
  print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Time: {epoch_duration:.2f}s") # Print duration

Epoch 1/10, Loss: 0.9078, Time: 245.81s
Epoch 2/10, Loss: 0.5947, Time: 261.66s
Epoch 3/10, Loss: 0.4651, Time: 263.46s
Epoch 4/10, Loss: 0.3759, Time: 262.12s
Epoch 5/10, Loss: 0.3091, Time: 262.42s
Epoch 6/10, Loss: 0.2720, Time: 262.59s
Epoch 7/10, Loss: 0.2402, Time: 262.92s
Epoch 8/10, Loss: 0.2090, Time: 262.50s
Epoch 9/10, Loss: 0.1894, Time: 261.53s
Epoch 10/10, Loss: 0.1772, Time: 262.38s


In [16]:
# Final evaluation train data
vgg16.eval()

total = 0
correct = 0

with torch.no_grad():
  for images, labels in train_dataloader:
    images = images.to(device)
    labels = labels.to(device)

    outputs= vgg16(images)
    _, predicted = torch.max(outputs.data, 1)

    total += labels.size(0)
    correct += (predicted == labels).sum().item()

final_accuracy = correct / total
print(f"Final Test Accuracy (best model): {final_accuracy:.4f}")

Final Test Accuracy (best model): 0.9908


In [17]:
# Final evaluation after loading the best model
vgg16.eval()

total = 0
correct = 0

with torch.no_grad():
  for images, labels in test_dataloader:
    images = images.to(device)
    labels = labels.to(device)

    outputs= vgg16(images)
    _, predicted = torch.max(outputs.data, 1)

    total += labels.size(0)
    correct += (predicted == labels).sum().item()

final_accuracy = correct / total
print(f"Final Test Accuracy (best model): {final_accuracy:.4f}")

Final Test Accuracy (best model): 0.8069
